In [1]:
import pandas as pd
from pathlib import Path

# Lokasi dataset
BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data" / "raw"

# Baca metadata
df = pd.read_csv(
    DATA_DIR / "validated.tsv",
    sep="\t"
)

# Tampilkan jumlah data
print("Jumlah seluruh klip:", len(df))

# Tampilkan kolom yang tersedia
print(df.columns.tolist())

# Tampilkan informasi kalimat tanpa membuka audio
display(
    df[[
        "path",
        "sentence",
        "up_votes",
        "down_votes",
        "client_id"
    ]].head(20)
)
df.shape

Jumlah seluruh klip: 30901
['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']


,path,sentence,up_votes,down_votes,client_id
0,common_voice_id_39599471.mp3,Dia tidak sepenuhnya mempercayaiku.,2,0,00f046cd1abd7e4ed354c2d97b67aceee8deb20bf53e5c...
1,common_voice_id_27657418.mp3,Berharap kau di sini.,2,0,025f07759891360d312521dd83679fa6a233b9d7aa9d4a...
2,common_voice_id_23033570.mp3,Maha Suci Allah.,2,0,0a4e7e60acc2a6145fc2589e6abb81e41e430cea0611bd...
3,common_voice_id_40088845.mp3,"Pada akhirnya, dia memilih anak kucing yang sa...",2,0,0a64c046530244b7507d4404575e8f673613c939ad6fac...
4,common_voice_id_36134676.mp3,Kucingku tersayang sudah seminggu hilang.,2,0,140e191f9fbdc062051af9265b4cf343b732fce3796051...
5,common_voice_id_20425643.mp3,Inilah dunia kecil.,2,1,19285f8e012ad31cad237d53bab348ce59a5cc13684754...
6,common_voice_id_32647488.mp3,"Ketika dia sampai stasiun, keretanya telah ber...",2,1,1d4e1d0cfef0c67c55f809a40973083c61c5fead9d2040...
7,common_voice_id_40538729.mp3,"Karena sibuk, saya tidak bisa makan siang.",2,0,26afbfd8031e3780a40beb6f46639cf7f189e80fadde4d...
8,common_voice_id_31865495.mp3,Dia dapat berenang dengan cepat.,2,1,27c9629f1250c9d4b8519bc72c079829b17f0bd8cd39b7...
9,common_voice_id_39610466.mp3,Bangunan besar itu gemerlap dengan cahaya.,2,0,28dafc8abefc6d6510821c3485070974e2b4ac6dc3c723...


(30901, 13)

In [2]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

display(df[["path", "sentence"]])

,path,sentence
0,common_voice_id_39599471.mp3,Dia tidak sepenuhnya mempercayaiku.
1,common_voice_id_27657418.mp3,Berharap kau di sini.
2,common_voice_id_23033570.mp3,Maha Suci Allah.
3,common_voice_id_40088845.mp3,"Pada akhirnya, dia memilih anak kucing yang satunya."
4,common_voice_id_36134676.mp3,Kucingku tersayang sudah seminggu hilang.
...,...,...
30896,common_voice_id_25970688.mp3,Sekolah ini memiliki empat laboratorium komputer plus perpustakaan.
30897,common_voice_id_25970689.mp3,Ada fazenda besar dan peternakan.
30898,common_voice_id_25970690.mp3,Normal untuk menganggapnya sebagai velositas dari beberapa hal fisik.
30899,common_voice_id_25970691.mp3,Saya harap saya tidak pernah mencemooh apa yang bijak dan baik.


In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw/")
CLIPS_DIR = DATA_DIR / "clips"

# 1. Baca metadata validated
df = pd.read_csv(
    DATA_DIR / "validated.tsv",
    sep="\t"
)

# 2. Filter berdasarkan validasi
df = df[
    (df["up_votes"] >= 2) &
    (df["up_votes"] > df["down_votes"]) &
    (df["sentence"].notna()) &
    (df["sentence"].str.strip() != "")
].copy()

# 3. Baca metadata durasi
duration_file = DATA_DIR / "clip_durations.tsv"

if duration_file.exists():
    durations = pd.read_csv(
        duration_file,
        sep="\t"
    )

    # Sesuaikan nama kolom durasi
    duration_col = [
        col for col in durations.columns
        if "duration" in col.lower()
    ][0]

    # Gabungkan berdasarkan nama file
    df = df.merge(
        durations,
        left_on="path",
        right_on="clip",
        how="left"
    )

    # Konversi milidetik menjadi detik
    df["duration_seconds"] = (
        df[duration_col] / 1000
    )

    df = df[
        df["duration_seconds"].between(5, 15)
    ]

# 4. Pastikan file audio tersedia
df["file_exists"] = df["path"].apply(
    lambda p: (CLIPS_DIR / p).exists()
)

df = df[df["file_exists"]].copy()

# 5. Urutkan berdasarkan kualitas validasi
df = df.sort_values(
    by=["up_votes", "down_votes"],
    ascending=[False, True]
)

print("Jumlah kandidat setelah seleksi:", len(df))

display(
    df[[
        "path",
        "sentence",
        "up_votes",
        "down_votes"
    ]].head(30)
)


Jumlah kandidat setelah seleksi: 7652


,path,sentence,up_votes,down_votes
9882,common_voice_id_35291053.mp3,Aku menyekakmat lawanku menggunakan kedua bentengku.,12,0
9919,common_voice_id_35277486.mp3,Saya kira kereta lebih cepat daripada bus sampai ke sekolah.,12,0
9920,common_voice_id_35277487.mp3,Beberapa kali mesin penjual otomatis itu tidak memberikan kembalian.,12,0
9933,common_voice_id_35277678.mp3,Banyak dari lagu religi di Ladino yang merupakan terjemahan dari bahasa Ibrani.,12,0
9940,common_voice_id_35284858.mp3,"Kemarin saya membeli sweter, rok, dan lain-lain di toserba.",12,0
10016,common_voice_id_35297877.mp3,"Bagi saya, tidak ada perbedaan mendasar antara kebudayaan dari kedua negara tersebut.",12,0
10029,common_voice_id_35304847.mp3,Saya berandai-andai apakah Tom masih mengingat bagaimana cara untuk melakukannya.,12,0
1292,common_voice_id_30077104.mp3,"Karena saya sudah mengambil banyak foto, saya tidak ada film lagi.",9,0
3004,common_voice_id_20879215.mp3,"Adalah penting, di usia berapapun, untuk takut akan Tuhan dan menghormati orang tua.",9,0
174,common_voice_id_35281927.mp3,Aku menyuruh adikku untuk membeli gula di warung.,8,0


In [4]:
import shutil
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "raw"
CLIPS_DIR = DATA_DIR / "clips"

OUTPUT_DIR = BASE_DIR / "data" / "selected"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ambil maksimal satu klip per penutur
candidates = df.drop_duplicates(
    subset="client_id"
).copy()

# Ambil 20 klip secara acak
selected = candidates.sample(
    n=min(20, len(candidates)),
    random_state=7
).reset_index(drop=True)

# Salin file audio terpilih
for i, row in selected.iterrows():

    source = CLIPS_DIR / row["path"]

    # Beri nama yang lebih mudah dikenali
    destination = OUTPUT_DIR / f"cv_{i+1:03d}{source.suffix}"

    shutil.copy2(source, destination)

    selected.loc[i, "selected_path"] = str(destination)

# Simpan metadata hasil seleksi
selected.to_csv(
    OUTPUT_DIR / "selected_metadata.csv",
    index=False
)

print("Jumlah klip terpilih:", len(selected))
print("Lokasi:", OUTPUT_DIR)

display(
    selected[[
        "selected_path",
        "sentence",
        "up_votes",
        "down_votes"
    ]]
)


Jumlah klip terpilih: 20
Lokasi: d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected


,selected_path,sentence,up_votes,down_votes
0,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_001.mp3,Apa yang membuat Anda memutuskan untuk bekerja di perusahaan kami?,2,0
1,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_002.mp3,Mengapa kalian tidak mau katakan kebenaran pada kami?,2,0
2,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_003.mp3,Saya diminta untuk meneruskan pesan ini kepadamu.,2,0
3,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_004.mp3,"Ketika saya datang ke sini, saya tidak kenal siapa pun.",2,0
4,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_005.mp3,Sepertinya kamu akan mendapatkan perkerjaan yang kamu impikan selama ini.,2,1
5,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_006.mp3,"Kalau jendela dibuka, hawanya akan sedikit sejuk.",2,0
6,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_007.mp3,Saya menulis nama dan alamat di formulir ini.,2,0
7,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_008.mp3,pantang menyerah salah satu kunci sukses,3,0
8,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_009.mp3,Saya mengajar bahasa Inggris kepada Pak Kimura.,2,0
9,d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_010.mp3,Apa Tom sudah tiba?,2,0


In [5]:
for i, row in selected.iterrows():
    print(f"Clip {i+1:02d}")
    print("File     :", row["selected_path"])
    print("Sentence :", row["sentence"] + "\n")

Clip 01
File     : d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_001.mp3
Sentence : Apa yang membuat Anda memutuskan untuk bekerja di perusahaan kami?

Clip 02
File     : d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_002.mp3
Sentence : Mengapa kalian tidak mau katakan kebenaran pada kami?

Clip 03
File     : d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_003.mp3
Sentence : Saya diminta untuk meneruskan pesan ini kepadamu.

Clip 04
File     : d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_004.mp3
Sentence : Ketika saya datang ke sini, saya tidak kenal siapa pun.

Clip 05
File     : d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_005.mp3
Sentence : Sepertinya kamu akan mendapatkan perkerjaan yang kamu impikan selama ini.

Clip 06
File     : d:\Kuliah\Pengolahan Sinyal Digital\PROJECT\data\selected\cv_006.mp3
Sentence : Kalau jendela dibuka, hawanya akan sedikit sejuk.

Clip 07
File     : d:\Kuliah\Pengolahan Sinyal Di

In [6]:
from IPython.display import Audio, display

audio_path = selected.iloc[7]["selected_path"]

display(Audio(filename=audio_path))